# Puente tipo Z24 en OpenSeesPy: frecuencias naturales

Modelo 2D de flexión vertical del puente sano. Solo geometría, propiedades, modelo y análisis de valores propios.

**Datos tomados de las fuentes**
- Vanos 14 + 30 + 14 m; ancho 8.60 m; canto de la viga 1.10 m; gálibo 4.50 m (Maeck y De Roeck, 2003, fig. 2).
- Extremos: columnas con rótulas de concreto → apoyo articulado. Pilas intermedias empotradas a la viga (Reynders y De Roeck, 2014).
- Espesor real del asfalto 16–18 cm (Reynders y De Roeck, 2014).
- Frecuencia medida de referencia, modo 1: 3.89 Hz (Maeck y De Roeck, 2003, tabla 2, referencia 2).

**Supuestos (`ASSUMED`)**: espesores de losas y almas, ancho de la losa inferior, módulo del concreto, densidades, sección y altura de las pilas. Ninguna fuente los da.

In [6]:
# Install once if needed: pip install openseespy
import numpy as np
import openseespy.opensees as ops

## 1. Parámetros

In [7]:
# --- Geometry (Maeck & De Roeck 2003) ---
spans = [14.0, 30.0, 14.0]     # m
deck_width = 8.60              # m, top slab width
deck_depth = 1.10              # m, total girder depth
pier_height = 4.50             # m, clearance used as pier height (ASSUMED)

# --- Box-girder cross-section (ASSUMED dimensions) ---
t_top = 0.25                   # m, top slab thickness
t_bot = 0.20                   # m, bottom slab thickness
b_bot = 5.00                   # m, bottom slab width
n_webs = 3                     # two-cell box -> three webs
t_web = 0.30                   # m, web thickness

# --- Materials (ASSUMED) ---
E_c = 32e9                     # Pa, concrete modulus
rho_c = 2500.0                 # kg/m^3, reinforced concrete density
rho_asph = 2300.0              # kg/m^3, asphalt density
t_asph = 0.17                  # m, asphalt thickness (16-18 cm, Reynders & De Roeck 2014)
b_asph = 8.00                  # m, paved width (ASSUMED)

# --- Piers clamped into the girder (ASSUMED section) ---
use_pier_clamping = True       # False -> simple supports at piers
pier_b = 0.80                  # m, pier thickness along bridge axis
pier_h = 4.00                  # m, pier width across the bridge

# --- Mesh ---
el_size = 1.0                  # m, deck element length

## 2. Propiedades de la sección

La sección cajón se divide en rectángulos (losa superior, losa inferior y almas). Para cada rectángulo $i$ de ancho $b_i$ y altura $h_i$:

$$
A=\sum_i A_i,\qquad \bar y=\frac{\sum_i A_i\,y_i}{A},\qquad
I=\sum_i\left(\frac{b_i h_i^3}{12}+A_i\,(y_i-\bar y)^2\right)
$$

- $A_i=b_i h_i$: área del rectángulo; $y_i$: altura de su centroide medida desde abajo.
- $\bar y$: centroide de la sección; $I$: inercia respecto al eje horizontal que pasa por el centroide (teorema de Steiner).

Masa por metro: $\bar m=\rho_c A+\rho_a\,t_a\,b_a$, donde $\rho_c$ y $\rho_a$ son las densidades del concreto y del asfalto, y $t_a$, $b_a$ el espesor y ancho del asfalto. El asfalto solo aporta masa, no rigidez.

In [8]:
h_web = deck_depth - t_top - t_bot
# (width, height, centroid height from bottom)
parts = [(deck_width, t_top, deck_depth - t_top / 2),
         (b_bot, t_bot, t_bot / 2),
         (n_webs * t_web, h_web, t_bot + h_web / 2)]

A = sum(b * h for b, h, _ in parts)
y_bar = sum(b * h * y for b, h, y in parts) / A
I = sum(b * h**3 / 12 + b * h * (y - y_bar)**2 for b, h, y in parts)
m_bar = rho_c * A + rho_asph * t_asph * b_asph

I_pier = pier_h * pier_b**3 / 12
A_pier = pier_h * pier_b

print(f"A = {A:.3f} m^2,  y_bar = {y_bar:.3f} m,  I = {I:.3f} m^4")
print(f"Mass per metre = {m_bar:,.0f} kg/m")

A = 3.735 m^2,  y_bar = 0.670 m,  I = 0.572 m^4
Mass per metre = 12,466 kg/m


## 3. Modelo en OpenSeesPy

- Plano, 3 grados de libertad por nodo: $u_x$, $u_y$ (vertical) y $\theta_z$.
- Viga: elementos `elasticBeamColumn` con masa consistente (`-cMass`).
- Apoyos extremos: $u_y$ restringido, giro libre (rótula).
- Pilas: $u_y$ restringido en la viga y, si `use_pier_clamping`, una columna empotrada en la base que aporta rigidez al giro.
- $u_x$ restringido en la viga para quedarse solo con modos de flexión vertical.

In [9]:
# Deck nodes
x_nodes = [0.0]
for L in spans:
    n = int(round(L / el_size))
    x_nodes += list(x_nodes[-1] + np.linspace(0, L, n + 1)[1:])
x_nodes = np.array(x_nodes)
support_x = np.cumsum([0.0] + spans)
support_nodes = [int(np.argmin(abs(x_nodes - s))) for s in support_x]
pier_nodes = support_nodes[1:-1]
n_deck = len(x_nodes)

ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 3)
for i, x in enumerate(x_nodes):
    ops.node(i + 1, float(x), 0.0)
    uy = 1 if i in support_nodes else 0
    ops.fix(i + 1, 1, uy, 0)                     # ux fixed: vertical bending only

ops.geomTransf("Linear", 1)
for i in range(n_deck - 1):
    ops.element("elasticBeamColumn", i + 1, i + 1, i + 2,
                A, E_c, I, 1, "-mass", m_bar, "-cMass")

if use_pier_clamping:
    for k, n in enumerate(pier_nodes):
        base = 1000 + k
        ops.node(base, float(x_nodes[n]), -pier_height)
        ops.fix(base, 1, 1, 1)                   # clamped foundation
        ops.element("elasticBeamColumn", 500 + k, base, n + 1,
                    A_pier, E_c, I_pier, 1)

print(f"Deck nodes: {n_deck}, supports at x = {x_nodes[support_nodes]}")

Deck nodes: 59, supports at x = [ 0. 14. 44. 58.]


## 4. Frecuencias naturales

$$
\mathbf{K}\,\boldsymbol\phi=\lambda\,\mathbf{M}\,\boldsymbol\phi,\qquad \lambda=\omega^2,\qquad f=\frac{\sqrt{\lambda}}{2\pi}
$$

- $\mathbf{K}$, $\mathbf{M}$: matrices de rigidez y masa; $\boldsymbol\phi$: forma modal.
- $\lambda$: valor propio; $\omega$: frecuencia circular (rad/s); $f$: frecuencia natural (Hz).

Referencias medidas para comparar (Maeck y De Roeck, 2003, tabla 2): modo 1 = 3.89 Hz (flexión simétrica); modos 3 y 4 = 9.80 y 10.30 Hz (flexión antisimétrica con torsión). El modo 2 medido (5.02 Hz) es lateral y este modelo no lo tiene.

In [10]:
n_modes = 3
lam = np.array(ops.eigen(n_modes))
freqs = np.sqrt(lam) / (2 * np.pi)

for k, f in enumerate(freqs, start=1):
    print(f"Mode {k} (vertical bending): {f:.2f} Hz")
print(f"\nMode 1 vs measured 3.89 Hz: {100 * (freqs[0] / 3.89 - 1):+.1f} %")

Mode 1 (vertical bending): 3.89 Hz
Mode 2 (vertical bending): 10.24 Hz
Mode 3 (vertical bending): 12.85 Hz

Mode 1 vs measured 3.89 Hz: +0.1 %
